# Stage 2 -- Classical and transformation baselines on S1 (part of E1, E7)
Runs RF-BR, LGBM-BR, CC-ens, RAkEL and ML-kNN on all three feature views under the S1
protocol, three seeds (one in quick mode). Thresholds tuned on validation only.
**Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`.

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage2_baselines')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

In [ ]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

In [ ]:
splits, sdf = scvd.load_splits()
if splits is None:
    log.warning('stage1 splits not found -- recomputing')
    splits = scvd.build_splits(pre, lab, seed=0)

In [ ]:
rows = []
res_path = f'{root}/results/stage2_baselines_results.csv'
done = set()
if os.path.exists(res_path):                    # resume after a kernel restart
    done = {tuple(r) for r in pd.read_csv(res_path)[['model', 'view', 'seed']].values}
    log.info('resuming: %d runs already checkpointed', len(done))
for view in scvd.VIEWS:
    d = scvd.make_data(pre, post, lab, fam, view, *splits['S1'])
    for model in ['RF-BR', 'LGBM-BR', 'CC-ens', 'RAkEL', 'ML-kNN']:
        for seed in SEEDS:
            if (model, view, seed) in done:
                log.info('skip %-8s F_%-4s seed%d (checkpointed)', model, view, seed)
                continue
            t0 = time.time()
            proba = scvd.fit_classical(model, d['X_tr'], d['y_tr'], seed)  # train ONCE
            th = scvd.tune_thresholds(d['y_va'], proba(d['X_va']))
            pt = proba(d['X_te'])
            m = scvd.all_metrics(d['y_te'], (pt >= th).astype(int), pt)
            row = {'model': model, 'view': view, 'split': 'S1', 'seed': seed,
                   'seconds': time.time() - t0, **m}
            rows.append(row)
            scvd.append_results(root, 'stage2_baselines', [row])   # checkpoint per run
            log.info('%-8s F_%-4s seed%d macro_f1=%.4f (%.0fs)', model, view, seed,
                     m['macro_f1'], row['seconds'])

In [ ]:
summ, sp = scvd.write_summary(root, 'stage2_baselines')
scvd.plot_model_macro_f1(summ, root, 's2_classical_macro_f1', log)
scvd.plot_perlabel_f1(summ, ['RF-BR', 'LGBM-BR', 'CC-ens'], root, 's2_classical_perlabel', log)
scvd.tex_table(summ, f'{root}/tables/stage2_baselines.tex',
               ['macro_f1', 'micro_f1', 'macro_ap', 'hamming'],
               'Classical and transformation baselines on S1 (mean over seeds).',
               'tab:stage2baselines')
summ.round(4)

In [ ]:
url = scvd.push_dataset(root, 'stage2_baselines', 'scvd-stage2-baselines', 'SCVD Stage 2: classical multi-label baselines on DIVE', log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)